# T-REX 한식 인식 모델 학습 (YOLOv8n → TFLite)

AI Hub **74번 "음식 이미지 및 영양정보 텍스트"**(400종, 박스 라벨 포함)로 YOLOv8n을 파인튜닝하고, 앱(`app/src/main/assets/models/`)에 넣을 수 있는 형태로 변환하는 노트북이다. 모델은 휴대폰 온디바이스(LiteRT INT8) 기준이라 YOLOv8n 을 유지한다.

### 데이터는 PC 에서 만들고, Colab 은 학습만 한다
AI Hub 는 **해외 IP 다운로드를 막는다**(Colab 에서 `aihubshell -mode d` 는 502 "해외에서의 데이터 다운로드를 제한"). 그래서 다운로드·전처리는 한국 PC 에서 `training/local_prep/`(pipeline.sh + prep_dataset.js)로 돌리고, 결과물 **`dataset.zip`(640px 이미지 + YOLO 라벨 + data.yaml + food_labels.txt + nutrition.json, 3~5GB)** 만 Google Drive `MyDrive/trex/dataset.zip` 에 올린다. 자세한 구조·근거는 `training/local_prep/README.md`.

### 실행 전 준비 (1회)
1. Drive `MyDrive/trex/dataset.zip` 이 올라가 있어야 한다.
2. Colab 런타임 유형: **G4 GPU + 고용량 RAM**(안 잡히면 A100 → L4). TPU 는 안 된다.
3. 아래 셀을 위에서부터 순서대로 실행.

### 산출물 (마지막 셀에서 zip으로 다운로드)
- `yolov8n_food.tflite` — INT8 양자화, **입출력은 float32 유지**. 앱 FoodDetector 는 NHWC/NCHW 입력을 모두 처리한다.
- `food_labels.txt` — 모델 클래스 인덱스 순서와 동일한 라벨 목록

두 파일을 `TREX_UI/app/src/main/assets/models/`에 덮어쓰고, `nutrition.json` 으로 `foodDatabase` 를 갱신한 뒤 빌드하면 앱이 실추론으로 전환된다.

In [ ]:
# 1. 환경 설치 + 배정된 런타임 확인
# Colab 은 세션마다 GPU 가 다르게 배정된다(G4/H100/A100/L4/T4). 무엇을 받았는지 여기서 확인한다 — 아래 설정은 어느 GPU 든 그대로 동작한다.
!pip install -q -U ultralytics
import torch, ultralytics
ultralytics.checks()
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, VRAM {p.total_memory / 1e9:.0f} GB')
else:
    print('⚠️ GPU 가 배정되지 않았다 — 런타임 → 런타임 유형 변경에서 GPU 를 고른다')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2
!df -h /content | tail -1

In [ ]:
# 2. Drive 연결 + PC 에서 만든 dataset 압축 해제
# (AI Hub 는 해외 IP 다운로드를 막으므로 Colab 에서 직접 받지 않는다 — 첫 셀 설명 참조)
from google.colab import drive
drive.mount('/content/drive')

# training/local_prep 의 finalize 결과를 묶어 올린 것. tar 를 쓴다 —
# Windows 에서 만든 zip 은 경로 구분자가 백슬래시라 리눅스 unzip 이 폴더로 풀지 못한다.
DATASET_ARCHIVE = '/content/drive/MyDrive/trex/dataset.tar'
RUNS_DIR = '/content/drive/MyDrive/trex/runs'   # 학습 결과 보존(세션이 끊겨도 RESUME 가능)

import os
assert os.path.exists(DATASET_ARCHIVE), f'{DATASET_ARCHIVE} 이 없다 — PC 에서 만든 dataset.tar 를 Drive 에 올린다'
!rm -rf /content/dataset && mkdir -p /content/dataset
if DATASET_ARCHIVE.endswith('.tar'):
    !tar -xf "{DATASET_ARCHIVE}" -C /content/dataset
else:
    !unzip -qo "{DATASET_ARCHIVE}" -d /content/dataset
# tar 를 어떻게 말았느냐에 따라 안에 폴더가 한 겹 더 있을 수 있다
# (`tar -cf x.tar dataset_final` 은 dataset_final/ 을, `tar -cf x.tar -C dataset_final .` 은 내용을 넣는다).
# 아래 셀들은 /content/dataset/data.yaml 을 기대하므로 여기서 맞춰 둔다.
import os, shutil, glob
if not os.path.exists('/content/dataset/data.yaml'):
    inner = [d for d in glob.glob('/content/dataset/*') if os.path.isdir(d) and os.path.exists(d + '/data.yaml')]
    assert len(inner) == 1, f'data.yaml 을 찾지 못했다. /content/dataset 안: {os.listdir("/content/dataset")}'
    print(f'tar 안에 폴더가 한 겹 더 있다 → {inner[0]} 의 내용을 끌어올린다')
    for item in os.listdir(inner[0]):
        shutil.move(os.path.join(inner[0], item), '/content/dataset/' + item)
    os.rmdir(inner[0])

assert os.path.exists('/content/dataset/data.yaml'), '압축은 풀렸는데 data.yaml 이 없다'
!ls /content/dataset && head -5 /content/dataset/data.yaml
!echo "train: $(ls /content/dataset/images/train | wc -l)개 / val: $(ls /content/dataset/images/val | wc -l)개 / 클래스: $(wc -l < /content/dataset/food_labels.txt)종"

In [ ]:
# 3. 학습 설정 — 필요하면 여기만 수정한다
# 모델은 YOLOv8n 고정: 휴대폰 온디바이스(LiteRT, INT8)에서 돌리는 것이 기준이라 더 큰 모델을 쓰지 않는다.
# 런타임은 G4(RTX PRO 6000, 96GB) > H100 > A100 > L4 > T4 순으로 빠르지만 아래 값은 어느 GPU 든 그대로 둔다.
MODEL = 'yolov8n.pt'
IMG_SIZE = 640                   # 앱 FoodDetector 전처리와 동일한 입력 크기 (데이터셋도 640px 로 만들어져 있다)
EPOCHS = 100                     # G4/A100 ≈ 1시간 안팎, L4 ≈ 2~3시간. patience 로 조기 종료된다.
PATIENCE = 20                    # 검증 mAP 가 20 에폭 동안 안 오르면 멈춘다
BATCH = 64                       # v8n@640 은 약 10GB 라 L4(24GB) 이상 어디서나 들어간다. T4(16GB) 에서 OOM 이면 32.
WORKERS = 8                      # 데이터 로더 스레드
CACHE = 'ram'                    # 640px 데이터셋 3~5GB 는 RAM 캐시가 가능하다(고용량 RAM 런타임). 메모리 부족이면 'disk'
SEED = 42

In [ ]:
# 4. 데이터셋 확인 — PC 의 prep_dataset.js finalize 가 만든 data.yaml / food_labels.txt 를 그대로 쓴다
from pathlib import Path

DATASET = Path('/content/dataset')
data_yaml = DATASET / 'data.yaml'
labels_txt = DATASET / 'food_labels.txt'

# 확인을 먼저 한다 — 파일을 먼저 읽으면 FileNotFoundError 가 나서 원인을 알기 어렵다.
assert data_yaml.exists(), 'data.yaml 이 없다 — 2번 셀(압축 해제)을 먼저 실행한다'
assert labels_txt.exists(), 'food_labels.txt 가 없다 — 2번 셀(압축 해제)을 먼저 실행한다'

class_names = [l.strip() for l in labels_txt.read_text(encoding='utf-8').splitlines() if l.strip()]

# data.yaml 의 names 개수와 food_labels.txt 줄 수가 어긋나면 클래스 인덱스가 밀린다.
# 그대로 학습하면 앱이 라벨을 잘못 매핑하는데, 그때는 이미 늦다.
import re
yaml_names = re.findall(r'^\s*\d+:\s*\S', data_yaml.read_text(encoding='utf-8'), re.M)
assert len(yaml_names) == len(class_names), (
    f'data.yaml 의 클래스 {len(yaml_names)}종과 food_labels.txt {len(class_names)}종이 다르다 — '
    'PC 에서 prep_dataset.js finalize 를 다시 돌린다'
)

# data.yaml 의 path 가 이 런타임 경로와 다르면 맞춘다
text = data_yaml.read_text(encoding='utf-8')
if not text.startswith(f'path: {DATASET}'):
    text = 'path: ' + str(DATASET) + '\n' + '\n'.join(l for l in text.splitlines() if not l.startswith('path:')) + '\n'
    data_yaml.write_text(text, encoding='utf-8')

n_train = len(list((DATASET / 'images' / 'train').glob('*.jpg')))
n_val = len(list((DATASET / 'images' / 'val').glob('*.jpg')))
print(f'클래스 {len(class_names)}종 / train {n_train}장 / val {n_val}장')
print('예시 클래스:', class_names[:8], '…')

In [ ]:
# 5. YOLOv8n 파인튜닝
# 세션이 끊겼으면 RESUME = True 로 바꿔 같은 셀을 다시 실행한다(가중치는 Drive 의 RUNS_DIR 에 있다).
from pathlib import Path
from ultralytics import YOLO

# 런타임이 끊기면 앞 셀의 변수가 사라진다. NameError 대신 무엇을 돌려야 하는지 알려 준다.
for _name, _cell in [('RUNS_DIR', '2'), ('data_yaml', '4'), ('MODEL', '3'), ('EPOCHS', '3')]:
    if _name not in globals():
        raise SystemExit(f'{_name} 이(가) 없다 — {_cell}번 셀을 먼저 실행한다 (세션이 끊기면 변수가 사라진다).')

RESUME = False
last = Path(RUNS_DIR) / 'food' / 'weights' / 'last.pt'
if RESUME and last.exists():
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMG_SIZE,
        batch=BATCH,
        workers=WORKERS,
        cache=CACHE,
        cos_lr=True,
        seed=SEED,
        project=RUNS_DIR,
        name='food',
        exist_ok=True,
    )
BEST = str(Path(RUNS_DIR) / 'food' / 'weights' / 'best.pt')
print('best:', BEST)

In [ ]:
# 6. 검증 지표 확인 (mAP50이 0.8 이상이면 쓸 만하다). 헷갈리는 클래스는 RUNS_DIR/food/confusion_matrix.png 로 본다.
metrics = YOLO(BEST).val(data=str(data_yaml))
print('mAP50:', round(metrics.box.map50, 3), '/ mAP50-95:', round(metrics.box.map, 3))

In [ ]:
# 7. TFLite INT8 변환 — 입출력은 float32로 유지된다 (Ultralytics 기본 동작, 앱 요구사항)
# export 에 nms 인자를 주지 않는다(기본값). 앱 FoodDetector 는 raw 출력 (N, 4+클래스수, 8400) 을 기대한다.
from pathlib import Path
from ultralytics import YOLO

YOLO(BEST).export(format='tflite', int8=True, imgsz=IMG_SIZE, data=str(data_yaml))
tflite_candidates = sorted(Path(BEST).parent.rglob('*int8*.tflite'))
assert tflite_candidates, 'INT8 tflite 산출물을 찾지 못했다 — export 로그를 확인'
TFLITE_PATH = tflite_candidates[0]
print('변환 완료:', TFLITE_PATH)

In [ ]:
# 8. 앱 호환성 자동 검증 — FoodDetector가 기대하는 조건과 대조
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

assert inp['dtype'] == np.float32, f"입력이 {inp['dtype']} — 앱은 float32 입력만 받는다 (FoodDetector 로드 시 거부됨)"
assert out['dtype'] == np.float32, f"출력이 {out['dtype']} — float32여야 한다"
# 앱은 NHWC [1,H,W,3] 과 NCHW [1,3,H,W] 를 모두 처리한다. 어느 쪽인지 여기서 확인해 둔다.
shape = list(inp['shape'])
if shape == [1, IMG_SIZE, IMG_SIZE, 3]:
    layout = 'NHWC'
elif shape == [1, 3, IMG_SIZE, IMG_SIZE]:
    layout = 'NCHW'
else:
    raise AssertionError(f"입력 형태 {shape} — [1,{IMG_SIZE},{IMG_SIZE},3] 또는 [1,3,{IMG_SIZE},{IMG_SIZE}] 이어야 한다")
nc = len(class_names)
assert nc + 4 in list(out['shape']), f"출력 형태 {out['shape']}가 클래스 수 {nc}(+4)와 맞지 않는다"
print(f"검증 통과 — 입력 {shape} float32 ({layout}), 출력 {out['shape']} (클래스 {nc}종)")

In [ ]:
# 9. 앱 배포용 파일 패키징 + 다운로드
import shutil
from google.colab import files

out_dir = Path('/content/app_assets')
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()
shutil.copy(TFLITE_PATH, out_dir / 'yolov8n_food.tflite')
# 클래스 인덱스 순서 그대로 — 앱 food_labels.txt 형식 (전부 음식이므로 '#' 접두어 없음)
(out_dir / 'food_labels.txt').write_text('\n'.join(class_names) + '\n', encoding='utf-8')
# 앱 foodDatabase 갱신용 1인분 영양값 (PC 의 finalize 가 만든 것)
if (DATASET / 'nutrition.json').exists():
    shutil.copy(DATASET / 'nutrition.json', out_dir / 'nutrition.json')

shutil.make_archive('/content/trex_food_model', 'zip', out_dir)
files.download('/content/trex_food_model.zip')
print('완료 — zip 안의 tflite·labels 를 TREX_UI/app/src/main/assets/models/ 에 덮어쓰고, nutrition.json 으로 foodDatabase 를 갱신한다')

### 적용 방법
1. 받은 `trex_food_model.zip`을 풀어 `yolov8n_food.tflite`, `food_labels.txt` 두 파일을 `TREX_UI/app/src/main/assets/models/`에 **덮어쓴다**.
2. `nutrition.json`(1인분 기준 실제 영양값)으로 `TrexData.kt` 의 `foodDatabase` 를 갱신한다 — 라벨명 = DB 키.
3. 앱을 빌드·설치하면 FoodDetector가 모델을 발견하고 실추론으로 동작한다. 로드 시 로그에 입력·출력 형태가 찍힌다.

### 런타임 안내 (Colab Pro+, 2026-09 기준)
- 런타임 유형 변경에서 고를 수 있는 GPU: **G4(NVIDIA RTX PRO 6000 Blackwell, 96GB — 가장 빠름)**, H100, A100(고용량 RAM 켜면 80GB), L4, T4. G4·H100 은 배정이 안 될 때가 있으니 **G4 → A100 → L4** 순으로 잡히는 것을 쓴다. 1번 셀 출력으로 무엇을 받았는지 확인한다.
- YOLOv8n 학습은 GPU 를 크게 타지 않는다 — G4 라고 배치를 키울 필요 없고 에폭당 시간만 줄어든다.
- Pro+ 는 최대 24시간 세션·백그라운드 실행을 지원하지만 컴퓨팅 단위가 소진되면 끊긴다. 결과는 Drive(`RUNS_DIR`)에 남으므로 5번 셀 `RESUME = True` 로 이어서 학습한다.

### 문제 해결
- **OOM**: 3번 셀 `BATCH = 32`(T4). 그래도 나면 `CACHE = 'disk'`.
- **GPU 세션 끊김**: 5번 셀 `RESUME = True` 후 재실행(가중치는 Drive 에 있다).
- **정확도 낮음**: `EPOCHS`를 늘리거나 PC 전처리에서 클래스를 줄여 다시 만든다. 헷갈리는 클래스는 6번 셀의 confusion matrix 로 확인.
- **8번 셀 assert 실패(입출력이 float32가 아님)**: 7번 셀에서 `int8=True`를 `int8=False`로 바꿔 float32로 다시 변환한다(파일 2~3배). 산출물 검색 패턴도 `*int8*.tflite` 대신 `*.tflite`로.
- **8번 셀에서 출력이 `[1, 300, 6]`**: NMS 가 포함된 형태로 export 된 것. 7번 셀 export 에 `nms=` 인자가 들어가 있지 않은지 확인하고 빼서 다시 변환한다.